In [ ]:
if 'spark' in globals():
    spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 16:26:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-07-13-Jul"
TABLE_NAME = "videos"

execution_date = "2026-07-13-Jul"
full_file_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [4]:
df_events = spark.read.parquet(full_file_path).cache()
df_events.createOrReplaceTempView("events")

26/08/12 16:26:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
df_silver = spark.sql("""
SELECT
    event_id,
    user_id,
    video_id,
    interaction_type,
    device_type,

    CAST(watch_time_sec AS INT) AS watch_time_sec,

    TO_TIMESTAMP(event_timestamp) AS event_timestamp,

    TO_DATE(TO_TIMESTAMP(event_timestamp)) AS event_date,

    HOUR(TO_TIMESTAMP(event_timestamp)) AS event_hour,

    (
        interaction_type IN ('play', 'complete')
        AND watch_time_sec IS NULL
    ) AS dq_missing_watch_time,

    (
        interaction_type IN ('like', 'pause')
        AND watch_time_sec IS NOT NULL
    ) AS dq_unexpected_watch_time,

    (
        watch_time_sec IS NOT NULL
        AND (
            CAST(watch_time_sec AS INT) <= 0
            OR CAST(watch_time_sec AS INT) > 120
        )
    ) AS dq_invalid_watch_time,

    (
        interaction_type IS NULL
        OR interaction_type NOT IN ('like', 'play', 'complete', 'pause')
    ) AS dq_invalid_interaction,

    (
        device_type IS NULL
        OR device_type NOT IN ('mobile', 'tv', 'web')
    ) AS dq_invalid_device,

    (
        TO_TIMESTAMP(event_timestamp) IS NULL
    ) AS dq_invalid_timestamp,

    CURRENT_DATE() AS ingestion_date,

    CONCAT(
        DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss'),
        '_kafka'
    ) AS batch_id

FROM events
""")

In [ ]:
# 1.Row count + event_id uniqueness

In [6]:
df_silver.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("event_id").alias("distinct_event_ids")
).show()

+----------+------------------+
|total_rows|distinct_event_ids|
+----------+------------------+
|      1000|              1000|
+----------+------------------+



In [9]:
df_silver.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+



In [ ]:
# 2.Required fields

In [10]:
df_silver.filter(
    F.col("event_id").isNull() |
    F.col("user_id").isNull() |
    F.col("video_id").isNull() |
    F.col("interaction_type").isNull() |
    F.col("device_type").isNull()
).count()

0

In [ ]:
# 3.Categorical values

In [11]:
df_silver.filter(
    ~F.col("interaction_type").isin(
        "like", "play", "complete", "pause"
    )
).count()

0

In [12]:
df_silver.filter(
    ~F.col("device_type").isin(
        "mobile", "tv", "web"
    )
).count()

0

In [ ]:
# 4.Timestamp + derived columns

In [13]:
df_silver.filter(
    F.col("event_timestamp").isNull()
).count()

0

In [14]:
df_silver.filter(
    F.col("event_date") != F.to_date("event_timestamp")
).count()

0

In [15]:
df_silver.filter(
    F.col("event_hour") != F.hour("event_timestamp")
).count()

0

In [ ]:
# 5.Watch time rules

In [16]:
df_silver.filter(
    F.col("interaction_type").isin("like", "pause") &
    F.col("watch_time_sec").isNotNull()
).count()

0

In [17]:
df_silver.filter(
    F.col("interaction_type").isin("play", "complete") &
    F.col("watch_time_sec").isNull()
).count()

0

In [ ]:
# 6.Data-quality flags

In [18]:
df_silver.filter(
    F.col("dq_missing_watch_time") != (
        F.col("interaction_type").isin("play", "complete") &
        F.col("watch_time_sec").isNull()
    )
).count()

0

In [19]:
df_silver.filter(
    F.col("dq_unexpected_watch_time") != (
        F.col("interaction_type").isin("like", "pause") &
        F.col("watch_time_sec").isNotNull()
    )
).count()

0

In [20]:
df_silver.filter(
    F.col("dq_invalid_watch_time") != (
        F.col("watch_time_sec").isNotNull() &
        (
            (F.col("watch_time_sec") <= 0) |
            (F.col("watch_time_sec") > 120)
        )
    )
).count()

0

In [21]:
df_silver.filter(
    F.col("dq_invalid_interaction") != (
        ~F.col("interaction_type").isin(
            "like", "play", "complete", "pause"
        )
    )
).count()

0

In [22]:
df_silver.filter(
    F.col("dq_invalid_device") != (
        ~F.col("device_type").isin(
            "mobile", "tv", "web"
        )
    )
).count()

0

In [23]:
df_silver.filter(
    F.col("dq_invalid_timestamp") !=
    F.col("event_timestamp").isNull()
).count()

0

In [ ]:
# 7.Metadata

In [24]:
df_silver.filter(
    F.col("ingestion_date").isNull() |
    F.col("batch_id").isNull()
).count()

0